# 📦 Notebook 1: Data Loading & Cleaning

This notebook serves as the entry point for the Sales Forecasting System by importing raw Excel data into a pandas environment. It performs essential cleaning steps and resamples the data into a weekly time series format ready for exploratory analysis.

# 01 Data Loading & Cleaning
This notebook handles the initial loading, cleaning, and weekly resampling of the sales forecasting dataset.

In this first code block, we import the necessary libraries for data manipulation (pandas), file system operations (os), and visualization (matplotlib and seaborn). We also set a default visual style for our charts.

In [ ]:
import pandas as pd # Used for data manipulation and analysis
import os # Provides functions for interacting with the operating system
import matplotlib.pyplot as plt # Core plotting library
import seaborn as sns # Statistical data visualization based on matplotlib

# Apply a clean, modern style to all future plots
plt.style.use('ggplot')
# Set a specific color palette for consistent aesthetics
sns.set_palette("viridis")

## Core Data Loading & Cleaning Pipeline
This logic is migrated from the original `DataLoader` and `preprocessing` scripts.

Here, we define a comprehensive function that reads the raw Excel file, converts dates into a standard format, fills in any missing values, and aggregates the data into weekly sums for each state.

In [ ]:
def load_and_clean_data(file_path):
    # 1. Load the raw Excel data into a DataFrame
    print(f"Loading data from {file_path}...")
    df = pd.read_excel(file_path, engine='openpyxl')
    
    # 2. Convert the 'Date' column to proper datetime objects for time-series analysis
    df['Date'] = pd.to_datetime(df['Date'])
    
    # 3. Set the 'Date' column as the index for easier resampling
    df.set_index('Date', inplace=True)
    
    # 4. Fill any missing values using forward-fill (carry previous sales forward)
    df = df.ffill()
    
    # 5. Resample the data to a Weekly frequency ('W') and sum the 'Total' sales for each state
    # This aggregates daily or irregular data into consistent weekly buckets
    cleaned_df = df.groupby('State').resample('W')['Total'].sum().reset_index()
    
    print(f"Successfully processed {len(cleaned_df)} weekly records across {len(cleaned_df['State'].unique())} states.")
    return cleaned_df

# Define the path to our source data file
data_path = "../data/Forecasting Case- Study.xlsx"

# Check if the file exists before attempting to load it
if os.path.exists(data_path):
    # Execute the loading and cleaning pipeline
    cleaned_data = load_and_clean_data(data_path)
    
    # Save the cleaned dataset to a CSV file for use in other notebooks
    cleaned_data.to_csv("../data/cleaned_data.csv", index=False)
    
    # Display the first few rows to confirm the structure looks correct
    print("\nSample Data:")
    print(cleaned_data.head())
else:
    # Print an error if the source file is missing
    print(f"Error: File not found at {data_path}")

## Quick Inspection

Finally, we perform a quick check of our processed data to verify the columns, count the number of states included, and look at basic statistics like mean and standard deviation.

In [ ]:
if 'cleaned_data' in locals():
    # List all column names to ensure 'State', 'Date', and 'Total' are present
    print("Columns:", cleaned_data.columns.tolist())
    
    # Count and display the number of unique states identified in the dataset
    print("\nState Count:", len(cleaned_data['State'].unique()))
    
    print("\nAll States:")
    print(cleaned_data['State'].unique())
    
    print("\nMissing Values Check (Should be all 0s):")
    print(cleaned_data.isnull().sum())
    
    # Show statistical summary (min, max, mean, count) for the sales volume
    print("\nStatistical Summary:")
    print(cleaned_data.describe())